In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

def entrenar_alden_gps():
    # 1. Cargar el dataset con las nuevas coordenadas
    # Asegúrate de que el archivo se llame exactamente así
    df = pd.read_csv('/content/drive/MyDrive/Mares del Pacifico LTDA/Vueltas - Hoja 1.csv')
    df.columns = [c.lower().strip() for c in df.columns]

    # 2. Limpieza de Texto
    for col in ['zona', 'embarcacion', 'centro']:
        df[col] = df[col].astype(str).str.lower().str.strip()

    # 3. Limpieza de Coordenadas (Maneja comas y puntos)
    df['latitud'] = df['latitud'].astype(str).str.replace(',', '.').astype(float)
    df['longitud'] = df['longitud'].astype(str).str.replace(',', '.').astype(float)

    # 4. Datos de Clima (Llenar vacíos)
    df['temp_aire'] = df['temp_aire'].fillna(df['temp_aire'].mean())
    df['lluvia_mm'] = df['lluvia_mm'].fillna(0)
    df['viento_nudos'] = df['viento_nudos'].fillna(df['viento_nudos'].median())

    # 5. Procesar Tiempos
    df['timestamp'] = pd.to_datetime(df['fecha'] + ' ' + df['hora_llegada'], dayfirst=True)
    df['dia_n'] = df['timestamp'].dt.weekday
    df['hora_n'] = df['timestamp'].dt.hour + (df['timestamp'].dt.minute / 60)

    # 6. Codificación (Convertir nombres a números para la IA)
    le_z, le_l, le_c = LabelEncoder(), LabelEncoder(), LabelEncoder()
    df['zona_n'] = le_z.fit_transform(df['zona'])
    df['lancha_n'] = le_l.fit_transform(df['embarcacion'])
    df['centro_n'] = le_c.fit_transform(df['centro'])

    # 7. Entrenamiento (Variables: Zona, Lancha, Centro, Hora, Día, Temp, Lluvia, Viento, Lat, Lon)
    features = ['zona_n', 'lancha_n', 'centro_n', 'hora_n', 'dia_n',
                'temp_aire', 'lluvia_mm', 'viento_nudos', 'latitud', 'longitud']
    X = df[features]
    y = df['gravedad']

    modelo = DecisionTreeClassifier(max_depth=8, random_state=42).fit(X, y)

    # 8. Crear Maestro de GPS automático
    maestro_gps = df[['centro', 'latitud', 'longitud']].drop_duplicates().set_index('centro').to_dict('index')

    # 9. Guardar todo el conocimiento
    joblib.dump({
        'modelo': modelo,
        'le_zona': le_z,
        'le_lancha': le_l,
        'le_centro': le_c,
        'maestro_gps': maestro_gps
    }, 'modelo_logistica_v3.pkl')

    print("✅ ENTRENAMIENTO EXITOSO")
    print(f"📍 Se registraron {len(maestro_gps)} centros con sus coordenadas.")

if __name__ == "__main__":
    entrenar_alden_gps()

✅ ENTRENAMIENTO EXITOSO
📍 Se registraron 48 centros con sus coordenadas.


In [ ]:
import joblib
import pandas as pd
import requests
from datetime import datetime, timedelta

def obtener_clima_alden(lat, lon, dias_offset, hora_obj):
    # API de clima gratuita
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&hourly=temperature_2m,rain,windspeed_10m&windspeed_unit=kn&timezone=America%2FSantiago&forecast_days=7"
    res = requests.get(url)
    if res.status_code == 200:
        d = res.json()['hourly']
        df_f = pd.DataFrame(d)
        df_f['time'] = pd.to_datetime(df_f['time'])

        target = (datetime.now() + timedelta(days=dias_offset)).replace(hour=hora_obj, minute=0, second=0)
        row = df_f.loc[(df_f['time'] - target).abs().idxmin()]

        # --- AJUSTE TIPO WINDY ---
        # Multiplicamos por 1.25 (factor de canal) y sumamos 2 nudos de seguridad
        v_real = (row['windspeed_10m'] * 1.25) + 2
        return row['temperature_2m'], row['rain'], v_real
    return 10.0, 0.0, 5.0

def iniciar_asistente():
    print("🔮 --- ALDEN: ASISTENTE DE LOGÍSTICA PREDICTIVA --- 🔮")

    try:
        # Cargar los datos entrenados
        data = joblib.load('modelo_logistica_v3.pkl')

        centro = input("⚓ Centro de destino: ").lower().strip()
        zona = input("📍 Zona (melinka/cisnes): ").lower().strip()
        lancha = input("🚤 Lancha: ").lower().strip()
        dia_op = int(input("📅 Día (0:Hoy, 1:Mañana, 2:Pasado): "))
        hora_op = int(input("⏰ Hora (0-23): "))

        # 1. Buscar GPS en el maestro guardado
        if centro in data['maestro_gps']:
            gps = data['maestro_gps'][centro]
            lat, lon = gps['latitud'], gps['longitud']

            # 2. Consultar clima futuro
            temp, lluvia, v_real = obtener_clima_alden(lat, lon, dia_op, hora_op)

            # 3. Traducir datos para el modelo
            z_n = data['le_zona'].transform([zona])[0]
            l_n = data['le_lancha'].transform([lancha])[0]
            c_n = data['le_centro'].transform([centro])[0]
            dia_semana = (datetime.now() + timedelta(days=dia_op)).weekday()

            # 4. Predecir Riesgo
            pred = data['modelo'].predict([[z_n, l_n, c_n, float(hora_op), dia_semana,
                                            temp, lluvia, v_real, lat, lon]])[0]

            niveles = {0:"✅ OK", 1:"🟢 MUY BAJO", 2:"🟡 BAJO", 3:"🟠 MEDIO", 4:"🔴 ALTO", 5:"💀 CRÍTICO"}

            # --- REPORTE FINAL ---
            print("\n" + "="*50)
            print(f"📍 GPS {centro.upper()}: {lat}, {lon}")
            print(f"🌬️ VIENTO WINDY-ADAPT: {v_real:.1f} kn")

            if v_real >= 15:
                print("🚫 AVISO: Posible cierre de puerto (>15 kn)")

            print("-" * 50)
            print(f"🔍 RIESGO LOGÍSTICO (ALDEN): {niveles.get(pred)}")
            print("="*50)
        else:
            print(f"❌ El centro '{centro}' no tiene coordenadas en el Excel.")

    except Exception as e:
        print(f"❌ Error: {e}")

if __name__ == "__main__":
    iniciar_asistente()

🔮 --- ALDEN: ASISTENTE DE LOGÍSTICA PREDICTIVA --- 🔮
